<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 18


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Review в C#, который будет представлять отзывы о
продуктах или услугах. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма.

1. ОтзывОбслуживания (ServiceReview): Должен содержать дополнительные
атрибуты, такие как Дата посещения (VisitDate).
Метод DisplayReview() должен быть переопределен для включения даты
посещения в отображение отзыва.
2. ОтзывТовара (ProductReview): Должен содержать дополнительные атрибуты,
такие как Идентификатор продукта (ProductId). Метод RateProduct() должен
быть переопределен для связывания рейтинга с конкретным продуктом.
3. ОтзывУслуги (ServiceReview) (если требуется третий класс): Должен
содержать дополнительные атрибуты, такие как Время начала услуги
(StartTime). Метод GetReviewDetails() должен быть переопределен для
отображения времени начала услуги вместе с другими деталями отзыва.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [6]:
var basic = new Review(1, "Отличный продукт", 4.5);
basic.DisplayReview();
basic.RateProduct(5.0);
basic.RateProduct(4.5, "Хороший продукт");
basic.RateProduct(4);
basic.AddAuthor("Артём");
basic.AddLike(3);
basic.VerifyReview();

var service = new ServiceReview(2, "Отличный сервис", 5.0, DateTime.Now);
service.SetEmployee("Иван", 5);
service.SetWaitingTime(45);

var product = new ProductReview(3, "Качественный товар", 4.8, "PROD-123");
product.SetBrand("Example");
product.RecommendProduct(false);

var detailed = new DetailedServiceReview(4, "Превосходное обслуживание", 4.9,
    DateTime.Now.AddDays(-3), DateTime.Now.AddHours(-5));
detailed.SetServiceType("Ремонт");
detailed.Price = 3000;
detailed.ApplyDiscount(10);
detailed.CalculateDuration(true);

var special = new SpecialReview(5, "Хороший товар", 5.0);
special.AddComment("Рекомендую!", "Артём");
special.ShareReview("Telegram");

var reviews = new List<Review> { basic, service, product, detailed, special };
foreach (Review r in reviews)
{
    r.DisplayReview();
    Console.WriteLine(r.GetReviewDetails());
    Console.WriteLine($"Положительный: {r.IsPositive()}\n");
}

var repository = new ReviewRepository<Review>();
reviews.ForEach(repository.Add);
Console.WriteLine($"Всего отзывов: {repository.Count}");
Console.WriteLine(repository.FindById(3)?.GetReviewDetails());
new ReviewResponse<string>(true, "Обработка завершена", "Успешно").Display();

IExportable exportable = detailed;
Console.WriteLine(exportable.Export());
IModerable moderable = special;
moderable.Approve();
moderable.Reject("Спам");

ILogger logger = new ConsoleLogger();
IReviewStorage storage = new InMemoryReviewStorage();
var reviewService = new ReviewService(storage, logger);
reviews.ForEach(reviewService.AddReview);
Console.WriteLine($"Средний рейтинг: {reviewService.GetAverageRating():F1}");

public interface ICommentable { void AddComment(string comment); }
public interface IShareable { void ShareReview(); }
public interface IExportable { string Export(); }
public interface IModerable { void Approve(); void Reject(string reason); }

public class Review : IExportable
{
    private int _reviewId;
    private string _text;
    private double _rating;

    public string Author { get; set; }
    public int Likes { get; set; }
    public int Views { get; set; }
    public bool IsVerified { get; set; }

    public int ReviewId { get { return _reviewId; } set { _reviewId = value; } }
    public string Text { get { return _text; } set { _text = value; } }
    public double Rating { get { return _rating; } set { _rating = ValidateRating(value); } }

    public Review(int reviewId, string text, double rating)
    {
        ReviewId = reviewId;
        Text = text;
        Rating = rating;
        Author = "Аноним";
    }

    protected double ValidateRating(double rating)
    {
        if (rating < 1 || rating > 5)
            throw new ArgumentException("Рейтинг должен быть от 1 до 5!");
        return rating;
    }

    public virtual void DisplayReview()
    {
        Views++;
        Console.WriteLine($"Отзыв #{ReviewId}: \"{Text}\"");
    }

    public virtual void RateProduct(double newRating)
    {
        Rating = newRating;
        Console.WriteLine($"Рейтинг: {Rating} из 5");
    }

    public void RateProduct(double newRating, string comment)
    {
        Rating = newRating;
        Console.WriteLine($"Рейтинг: {Rating} из 5. Комментарий: {comment}");
    }

    public void RateProduct(int stars)
    {
        Rating = stars;
        Console.WriteLine($"Установлено звёзд: {stars}");
    }

    public virtual string GetReviewDetails() =>
        $"ID: {ReviewId}, Текст: {Text}, Рейтинг: {Rating}, Автор: {Author}, " +
        $"Лайки: {Likes}, Просмотры: {Views}, Проверен: {IsVerified}";

    public virtual bool IsPositive() => Rating >= 4;

    public void AddLike() => Likes++;

    public void AddLike(int count)
    {
        if (count < 0) throw new ArgumentException("Лайки не могут быть отрицательными!");
        Likes += count;
    }

    public void VerifyReview() => IsVerified = !IsVerified;
    public void AddAuthor(string author) => Author = author;

    string IExportable.Export() => $"[EXPORT] {GetReviewDetails()}";
}

public class ServiceReview : Review
{
    private DateTime _visitDate;

    public string EmployeeName { get; set; }
    public double ServiceQuality { get; set; }
    public int WaitingMinutes { get; set; }

    public DateTime VisitDate { get { return _visitDate; } set { _visitDate = value; } }

    public ServiceReview(int reviewId, string text, double rating, DateTime visitDate)
        : base(reviewId, text, rating)
    {
        VisitDate = visitDate;
        EmployeeName = "Не указан";
        ServiceQuality = 5.0;
    }

    public void SetEmployee(string name) => EmployeeName = name;

    public void SetEmployee(string name, int quality)
    {
        if (quality < 1 || quality > 5) throw new ArgumentException("Качество от 1 до 5!");
        EmployeeName = name;
        ServiceQuality = quality;
    }

    public void SetWaitingTime(int minutes) => WaitingMinutes = minutes;
    public bool IsLongWait() => WaitingMinutes > 30;

    public override void DisplayReview()
    {
        Views++;
        Console.WriteLine($"Отзыв #{ReviewId}: \"{Text}\" Дата посещения: {VisitDate:dd.MM.yyyy}");
    }

    public override string GetReviewDetails() =>
        base.GetReviewDetails() +
        $", Посещение: {VisitDate:dd.MM.yyyy}, Сотрудник: {EmployeeName}, Ожидание: {WaitingMinutes} мин";

    public override bool IsPositive() => Rating >= 4 && ServiceQuality >= 3;
}

public class ProductReview : Review
{
    private string _productId;

    public string Brand { get; set; }
    public bool Recommended { get; set; }
    public int DeliveryDays { get; set; }

    public string ProductId { get { return _productId; } set { _productId = value; } }

    public ProductReview(int reviewId, string text, double rating, string productId)
        : base(reviewId, text, rating)
    {
        ProductId = productId;
        Brand = "Не указан";
        Recommended = true;
    }

    public void SetBrand(string brand) => Brand = brand;
    public bool IsFastDelivery() => DeliveryDays <= 3;
    public void RecommendProduct() => Recommended = true;
    public void RecommendProduct(bool recommended) => Recommended = recommended;

    public override void RateProduct(double newRating)
    {
        Rating = newRating;
        Console.WriteLine($"Продукт {ProductId} получил рейтинг: {Rating} из 5");
    }

    public override string GetReviewDetails() =>
        base.GetReviewDetails() + $", Товар: {ProductId}, Бренд: {Brand}, Рекомендован: {Recommended}";

    public override bool IsPositive() => Rating >= 4 && Recommended;
}

public class DetailedServiceReview : ServiceReview
{
    private DateTime _startTime;

    public DateTime EndTime { get; set; }
    public string ServiceType { get; set; }
    public double Price { get; set; }

    public DateTime StartTime { get { return _startTime; } set { _startTime = value; } }

    public DetailedServiceReview(int reviewId, string text, double rating,
        DateTime visitDate, DateTime startTime)
        : base(reviewId, text, rating, visitDate)
    {
        StartTime = startTime;
        EndTime = startTime.AddHours(1);
        ServiceType = "Стандартная услуга";
    }

    public void SetServiceType(string type) => ServiceType = type;

    public void ApplyDiscount(int percent)
    {
        if (percent < 0 || percent > 100) throw new ArgumentException("Скидка от 0 до 100%!");
        Price -= Price * percent / 100;
    }

    public void CalculateDuration() =>
        Console.WriteLine($"Продолжительность: {(EndTime - StartTime).TotalMinutes:F0} минут");

    public double CalculateDuration(bool showResult)
    {
        double minutes = (EndTime - StartTime).TotalMinutes;
        if (showResult) Console.WriteLine($"Продолжительность: {minutes:F0} минут");
        return minutes;
    }

    public override string GetReviewDetails() =>
        base.GetReviewDetails() +
        $", Начало: {StartTime:HH:mm}, Тип: {ServiceType}, Стоимость: {Price}";
}

public class SpecialReview : Review, ICommentable, IShareable, IModerable
{
    public string Comment { get; set; }
    public int Shares { get; set; }
    public string Platform { get; set; }
    public string Moderator { get; set; }

    public SpecialReview(int reviewId, string text, double rating)
        : base(reviewId, text, rating)
    {
        Comment = "";
        Platform = "Не указана";
        Moderator = "Не назначен";
    }

    public void AddComment(string comment)
    {
        Comment = comment;
        Console.WriteLine($"Комментарий: {Comment}");
    }

    public void AddComment(string comment, string author)
    {
        Comment = comment;
        Console.WriteLine($"Комментарий от {author}: {Comment}");
    }

    public void ShareReview()
    {
        Shares++;
        Console.WriteLine($"Отзыв опубликован {Shares} раз.");
    }

    public void ShareReview(string platform)
    {
        Platform = platform;
        Shares++;
        Console.WriteLine($"Отзыв опубликован на платформе {Platform}");
    }

    void IModerable.Approve()
    {
        Moderator = "Система";
        Console.WriteLine($"Отзыв #{ReviewId} одобрен ({Moderator}).");
    }

    void IModerable.Reject(string reason)
    {
        Moderator = "Система";
        Console.WriteLine($"Отзыв #{ReviewId} отклонён. Причина: {reason}");
    }

    public override void DisplayReview()
    {
        base.DisplayReview();
        Console.WriteLine($"Специальный отзыв. Платформа: {Platform}");
    }
}

public class ReviewRepository<T> where T : Review
{
    private readonly List<T> _reviews = new List<T>();

    public int Count => _reviews.Count;
    public void Add(T review) => _reviews.Add(review);
    public T FindById(int id) => _reviews.Find(r => r.ReviewId == id);
    public IEnumerable<T> GetAll() => _reviews;
}

public class ReviewResponse<T>
{
    public bool Success { get; set; }
    public string Message { get; set; }
    public T Data { get; set; }

    public ReviewResponse(bool success, string message, T data)
    {
        Success = success;
        Message = message;
        Data = data;
    }

    public void Display() =>
        Console.WriteLine($"Успех: {Success}, Сообщение: {Message}, Данные: {Data}");
}

public interface ILogger { void Log(string message); }

public interface IReviewStorage
{
    void Add(Review review);
    Review FindById(int id);
    IEnumerable<Review> GetAll();
}

public class ConsoleLogger : ILogger
{
    public void Log(string message) => Console.WriteLine(message);
}

public class InMemoryReviewStorage : ReviewRepository<Review>, IReviewStorage { }

public class ReviewService
{
    private readonly IReviewStorage _storage;
    private readonly ILogger _logger;

    public ReviewService(IReviewStorage storage, ILogger logger)
    {
        _storage = storage;
        _logger = logger;
    }

    public void AddReview(Review review)
    {
        _storage.Add(review);
        _logger.Log($"Добавлен отзыв #{review.ReviewId}");
    }

    public double GetAverageRating() =>
        _storage.GetAll().Any() ? _storage.GetAll().Average(r => r.Rating) : 0;
}

Отзыв #1: "Отличный продукт"
Рейтинг: 5 из 5
Рейтинг: 4,5 из 5. Комментарий: Хороший продукт
Установлено звёзд: 4
Продолжительность: 60 минут
Комментарий от Артём: Рекомендую!
Отзыв опубликован на платформе Telegram
Отзыв #1: "Отличный продукт"
ID: 1, Текст: Отличный продукт, Рейтинг: 4, Автор: Артём, Лайки: 3, Просмотры: 2, Проверен: True
Положительный: True

Отзыв #2: "Отличный сервис" Дата посещения: 05.11.2026
ID: 2, Текст: Отличный сервис, Рейтинг: 5, Автор: Аноним, Лайки: 0, Просмотры: 1, Проверен: False, Посещение: 05.11.2026, Сотрудник: Иван, Ожидание: 45 мин
Положительный: True

Отзыв #3: "Качественный товар"
ID: 3, Текст: Качественный товар, Рейтинг: 4,8, Автор: Аноним, Лайки: 0, Просмотры: 1, Проверен: False, Товар: PROD-123, Бренд: Example, Рекомендован: False
Положительный: False

Отзыв #4: "Превосходное обслуживание" Дата посещения: 02.11.2026
ID: 4, Текст: Превосходное обслуживание, Рейтинг: 4,9, Автор: Аноним, Лайки: 0, Просмотры: 1, Проверен: False, Посещение: 02.11.20